In [ ]:
from google.colab import files
from pathlib import Path
import hashlib

EXPECTED_FILE_NAME = "real_dataset_120_v1.zip"
EXPECTED_SHA256 = "B30688ADE5E8FDB832D4940D5E02D804B7B724EDA0CF51389AB64025F7B89A71"

uploaded_files = files.upload()

zip_path = Path("/content") / EXPECTED_FILE_NAME

if not zip_path.exists():
    raise FileNotFoundError(
        f"Expected file was not uploaded: {EXPECTED_FILE_NAME}"
    )

sha256_hash = hashlib.sha256(zip_path.read_bytes()).hexdigest().upper()

print(f"FILE_NAME={zip_path.name}")
print(f"FILE_SIZE_KB={zip_path.stat().st_size / 1024:.1f}")
print(f"SHA256={sha256_hash}")
print(f"HASH_MATCH={sha256_hash == EXPECTED_SHA256}")
print(f"UPLOAD_READY={zip_path.exists() and sha256_hash == EXPECTED_SHA256}")

Saving real_dataset_120_v1.zip to real_dataset_120_v1.zip
FILE_NAME=real_dataset_120_v1.zip
FILE_SIZE_KB=258.2
SHA256=B30688ADE5E8FDB832D4940D5E02D804B7B724EDA0CF51389AB64025F7B89A71
HASH_MATCH=True
UPLOAD_READY=True


In [ ]:
from pathlib import Path
import json
import zipfile
from collections import Counter

ZIP_PATH = Path("/content/real_dataset_120_v1.zip")
EXTRACT_PATH = Path("/content/real_dataset_120_v1")

if EXTRACT_PATH.exists():
    import shutil
    shutil.rmtree(EXTRACT_PATH)

with zipfile.ZipFile(ZIP_PATH, "r") as zip_file:
    zip_file.extractall(EXTRACT_PATH)

raw_dataset_path = EXTRACT_PATH / "dataset" / "raw"
config_path = EXTRACT_PATH / "dataset_config.json"

if not raw_dataset_path.exists():
    raise FileNotFoundError(
        f"Raw dataset folder not found: {raw_dataset_path}"
    )

if not config_path.exists():
    raise FileNotFoundError(
        f"Dataset configuration not found: {config_path}"
    )

json_files = sorted(raw_dataset_path.glob("*.json"))
label_counts = Counter()
invalid_files = []

for json_file in json_files:
    try:
        with json_file.open("r", encoding="utf-8") as file:
            capture = json.load(file)

        label = capture.get("label")
        samples = capture.get("samples", [])
        sample_count = capture.get("sample_count")

        if label not in {"left", "right", "up"}:
            invalid_files.append(
                f"{json_file.name}: invalid label {label}"
            )
            continue

        if sample_count != 100 or len(samples) != 100:
            invalid_files.append(
                f"{json_file.name}: sample_count={sample_count}, "
                f"actual_samples={len(samples)}"
            )
            continue

        label_counts[label] += 1

    except Exception as error:
        invalid_files.append(
            f"{json_file.name}: {type(error).__name__}: {error}"
        )

with config_path.open("r", encoding="utf-8") as file:
    dataset_config = json.load(file)

print(f"EXTRACT_PATH={EXTRACT_PATH}")
print(f"JSON_FILES={len(json_files)}")
print(f"LEFT={label_counts['left']}")
print(f"RIGHT={label_counts['right']}")
print(f"UP={label_counts['up']}")
print(f"INVALID_FILES={len(invalid_files)}")
print(f"CONFIG_LABELS={dataset_config.get('labels')}")
print(f"CONFIG_CHANNELS={dataset_config.get('channels')}")
print(f"CONFIG_INPUT_SHAPE={dataset_config.get('input_shape')}")

dataset_ready = (
    len(json_files) == 120
    and label_counts["left"] == 40
    and label_counts["right"] == 40
    and label_counts["up"] == 40
    and len(invalid_files) == 0
)

print(f"DATASET_READY={dataset_ready}")

if invalid_files:
    print("\nINVALID FILE DETAILS:")
    for invalid_file in invalid_files:
        print(invalid_file)

FileNotFoundError: Raw dataset folder not found: /content/real_dataset_120_v1/dataset/raw

In [ ]:
from pathlib import Path
import zipfile

ZIP_PATH = Path("/content/real_dataset_120_v1.zip")

with zipfile.ZipFile(ZIP_PATH, "r") as zip_file:
    entry_names = zip_file.namelist()

print(f"ZIP_ENTRIES={len(entry_names)}")
print("\nFIRST_10_ENTRIES:")

for entry_name in entry_names[:10]:
    print(repr(entry_name))

ZIP_ENTRIES=121

FIRST_10_ENTRIES:
'dataset\\raw\\20260730T015922_680818Z_left.json'
'dataset\\raw\\20260730T020134_560254Z_right.json'
'dataset\\raw\\20260730T204322_651750Z_left.json'
'dataset\\raw\\20260730T204331_724802Z_right.json'
'dataset\\raw\\20260730T205335_763130Z_left.json'
'dataset\\raw\\20260730T205403_981320Z_right.json'
'dataset\\raw\\20260730T205436_306671Z_left.json'
'dataset\\raw\\20260730T205510_208260Z_right.json'
'dataset\\raw\\20260730T205550_141732Z_left.json'
'dataset\\raw\\20260730T205557_001012Z_right.json'


In [ ]:
from pathlib import Path
import json
import shutil
import zipfile
from collections import Counter

ZIP_PATH = Path("/content/real_dataset_120_v1.zip")
EXTRACT_PATH = Path("/content/real_dataset_120_v1")

if EXTRACT_PATH.exists():
    shutil.rmtree(EXTRACT_PATH)

EXTRACT_PATH.mkdir(parents=True, exist_ok=True)

normalized_backslash_entries = 0

with zipfile.ZipFile(ZIP_PATH, "r") as zip_file:
    for zip_entry in zip_file.infolist():
        normalized_name = zip_entry.filename.replace("\\", "/")

        if "\\" in zip_entry.filename:
            normalized_backslash_entries += 1

        destination_path = EXTRACT_PATH / normalized_name

        # Prevent unsafe paths from escaping the extraction folder.
        if EXTRACT_PATH.resolve() not in destination_path.resolve().parents:
            raise RuntimeError(
                f"Unsafe ZIP entry detected: {zip_entry.filename}"
            )

        if zip_entry.is_dir():
            destination_path.mkdir(parents=True, exist_ok=True)
            continue

        destination_path.parent.mkdir(parents=True, exist_ok=True)

        with zip_file.open(zip_entry, "r") as source_file:
            with destination_path.open("wb") as destination_file:
                shutil.copyfileobj(source_file, destination_file)

raw_dataset_path = EXTRACT_PATH / "dataset" / "raw"
config_path = EXTRACT_PATH / "dataset_config.json"

if not raw_dataset_path.exists():
    raise FileNotFoundError(
        f"Raw dataset folder not found: {raw_dataset_path}"
    )

if not config_path.exists():
    raise FileNotFoundError(
        f"Dataset configuration not found: {config_path}"
    )

json_files = sorted(raw_dataset_path.glob("*.json"))
label_counts = Counter()
invalid_files = []

for json_file in json_files:
    try:
        with json_file.open("r", encoding="utf-8") as file:
            capture = json.load(file)

        label = capture.get("label")
        samples = capture.get("samples", [])
        sample_count = capture.get("sample_count")

        if label not in {"left", "right", "up"}:
            invalid_files.append(
                f"{json_file.name}: invalid label={label}"
            )
            continue

        if sample_count != 100 or len(samples) != 100:
            invalid_files.append(
                f"{json_file.name}: sample_count={sample_count}, "
                f"actual_samples={len(samples)}"
            )
            continue

        label_counts[label] += 1

    except Exception as error:
        invalid_files.append(
            f"{json_file.name}: {type(error).__name__}: {error}"
        )

with config_path.open("r", encoding="utf-8") as file:
    dataset_config = json.load(file)

dataset_ready = (
    len(json_files) == 120
    and label_counts["left"] == 40
    and label_counts["right"] == 40
    and label_counts["up"] == 40
    and len(invalid_files) == 0
)

print(f"NORMALIZED_BACKSLASH_ENTRIES={normalized_backslash_entries}")
print(f"JSON_FILES={len(json_files)}")
print(f"LEFT={label_counts['left']}")
print(f"RIGHT={label_counts['right']}")
print(f"UP={label_counts['up']}")
print(f"INVALID_FILES={len(invalid_files)}")
print(f"CONFIG_LABELS={dataset_config.get('labels')}")
print(f"CONFIG_CHANNELS={dataset_config.get('channels')}")
print(f"CONFIG_INPUT_SHAPE={dataset_config.get('input_shape')}")
print(f"DATASET_READY={dataset_ready}")

if invalid_files:
    print("\nINVALID FILE DETAILS:")
    for invalid_file in invalid_files:
        print(invalid_file)

NORMALIZED_BACKSLASH_ENTRIES=120
JSON_FILES=120
LEFT=40
RIGHT=40
UP=40
INVALID_FILES=0
CONFIG_LABELS=None
CONFIG_CHANNELS=None
CONFIG_INPUT_SHAPE=None
DATASET_READY=True


In [ ]:
import json

with config_path.open("r", encoding="utf-8") as file:
    dataset_config = json.load(file)

print("CONFIG_ROOT_TYPE=", type(dataset_config).__name__)
print("CONFIG_ROOT_KEYS=", list(dataset_config.keys()))
print("\nFULL_DATASET_CONFIG:")
print(json.dumps(dataset_config, indent=2))

CONFIG_ROOT_TYPE= dict
CONFIG_ROOT_KEYS= ['project_name', 'dataset_version', 'gesture_labels', 'sensor_channels', 'sensor_value_format', 'sample_rate_hz', 'sample_interval_ms', 'capture_duration_ms', 'samples_per_capture', 'minimum_captures_per_label', 'target_captures_per_label', 'orientation_rule', 'capture_rule', 'storage_directory']

FULL_DATASET_CONFIG:
{
  "project_name": "Wireless Embedded Gesture Recognition System",
  "dataset_version": "1.0",
  "gesture_labels": [
    "left",
    "right",
    "up"
  ],
  "sensor_channels": [
    "ax",
    "ay",
    "az",
    "gx",
    "gy",
    "gz"
  ],
  "sensor_value_format": "raw_int16",
  "sample_rate_hz": 50,
  "sample_interval_ms": 20,
  "capture_duration_ms": 2000,
  "samples_per_capture": 100,
  "minimum_captures_per_label": 30,
  "target_captures_per_label": 40,
  "orientation_rule": "Keep the MPU6050 in the same hand and physical orientation for every capture.",
  "capture_rule": "Press the button, perform one complete gesture duri

In [ ]:
from pathlib import Path
import json
import numpy as np
from collections import Counter

GESTURE_LABELS = dataset_config["gesture_labels"]
SENSOR_CHANNELS = dataset_config["sensor_channels"]
SAMPLES_PER_CAPTURE = dataset_config["samples_per_capture"]

label_to_index = {
    label: index
    for index, label in enumerate(GESTURE_LABELS)
}

X_samples = []
y_labels = []
loaded_file_names = []

for json_file in sorted(raw_dataset_path.glob("*.json")):
    with json_file.open("r", encoding="utf-8") as file:
        capture = json.load(file)

    label = capture["label"]
    samples = capture["samples"]

    if label not in label_to_index:
        raise ValueError(
            f"Unknown label in {json_file.name}: {label}"
        )

    if len(samples) != SAMPLES_PER_CAPTURE:
        raise ValueError(
            f"Invalid sample count in {json_file.name}: "
            f"{len(samples)}"
        )

    capture_matrix = np.zeros(
        (SAMPLES_PER_CAPTURE, len(SENSOR_CHANNELS)),
        dtype=np.float32
    )

    for sample_index, sample in enumerate(samples):
        missing_channels = [
            channel
            for channel in SENSOR_CHANNELS
            if channel not in sample
        ]

        if missing_channels:
            raise KeyError(
                f"Missing channels in {json_file.name}, "
                f"sample {sample_index}: {missing_channels}"
            )

        capture_matrix[sample_index] = [
            sample[channel]
            for channel in SENSOR_CHANNELS
        ]

    X_samples.append(capture_matrix)
    y_labels.append(label_to_index[label])
    loaded_file_names.append(json_file.name)

X = np.asarray(X_samples, dtype=np.float32)
y = np.asarray(y_labels, dtype=np.int64)

label_counts = Counter(
    GESTURE_LABELS[label_index]
    for label_index in y
)

expected_shape = (
    120,
    SAMPLES_PER_CAPTURE,
    len(SENSOR_CHANNELS)
)

print(f"GESTURE_LABELS={GESTURE_LABELS}")
print(f"SENSOR_CHANNELS={SENSOR_CHANNELS}")
print(f"LABEL_TO_INDEX={label_to_index}")
print(f"X_SHAPE={X.shape}")
print(f"Y_SHAPE={y.shape}")
print(f"X_DTYPE={X.dtype}")
print(f"Y_DTYPE={y.dtype}")
print(f"LEFT={label_counts['left']}")
print(f"RIGHT={label_counts['right']}")
print(f"UP={label_counts['up']}")
print(f"FINITE_VALUES={np.isfinite(X).all()}")
print(f"EXPECTED_SHAPE_MATCH={X.shape == expected_shape}")
print(
    "NUMPY_DATASET_READY="
    f"{X.shape == expected_shape and np.isfinite(X).all()}"
)

GESTURE_LABELS=['left', 'right', 'up']
SENSOR_CHANNELS=['ax', 'ay', 'az', 'gx', 'gy', 'gz']
LABEL_TO_INDEX={'left': 0, 'right': 1, 'up': 2}
X_SHAPE=(120, 100, 6)
Y_SHAPE=(120,)
X_DTYPE=float32
Y_DTYPE=int64
LEFT=40
RIGHT=40
UP=40
FINITE_VALUES=True
EXPECTED_SHAPE_MATCH=True
NUMPY_DATASET_READY=True


In [ ]:
import numpy as np
from collections import Counter

RANDOM_SEED = 42

TRAIN_CAPTURES_PER_LABEL = 24
VALIDATION_CAPTURES_PER_LABEL = 8
TEST_CAPTURES_PER_LABEL = 8

random_generator = np.random.default_rng(RANDOM_SEED)

train_indices = []
validation_indices = []
test_indices = []

for label_index, label_name in enumerate(GESTURE_LABELS):
    class_indices = np.where(y == label_index)[0]

    expected_class_count = (
        TRAIN_CAPTURES_PER_LABEL
        + VALIDATION_CAPTURES_PER_LABEL
        + TEST_CAPTURES_PER_LABEL
    )

    if len(class_indices) != expected_class_count:
        raise ValueError(
            f"Unexpected number of captures for {label_name}: "
            f"{len(class_indices)}"
        )

    shuffled_indices = random_generator.permutation(class_indices)

    train_end = TRAIN_CAPTURES_PER_LABEL
    validation_end = (
        TRAIN_CAPTURES_PER_LABEL
        + VALIDATION_CAPTURES_PER_LABEL
    )

    train_indices.extend(shuffled_indices[:train_end])
    validation_indices.extend(
        shuffled_indices[train_end:validation_end]
    )
    test_indices.extend(shuffled_indices[validation_end:])

train_indices = random_generator.permutation(train_indices)
validation_indices = random_generator.permutation(validation_indices)
test_indices = random_generator.permutation(test_indices)

X_train = X[train_indices]
y_train = y[train_indices]

X_validation = X[validation_indices]
y_validation = y[validation_indices]

X_test = X[test_indices]
y_test = y[test_indices]

train_file_names = np.asarray(loaded_file_names)[train_indices]
validation_file_names = np.asarray(loaded_file_names)[validation_indices]
test_file_names = np.asarray(loaded_file_names)[test_indices]

train_index_set = set(train_indices.tolist())
validation_index_set = set(validation_indices.tolist())
test_index_set = set(test_indices.tolist())

no_overlap = (
    train_index_set.isdisjoint(validation_index_set)
    and train_index_set.isdisjoint(test_index_set)
    and validation_index_set.isdisjoint(test_index_set)
)

all_samples_assigned = (
    train_index_set
    | validation_index_set
    | test_index_set
) == set(range(len(X)))

def get_label_counts(label_array):
    return Counter(
        GESTURE_LABELS[label_index]
        for label_index in label_array
    )

train_counts = get_label_counts(y_train)
validation_counts = get_label_counts(y_validation)
test_counts = get_label_counts(y_test)

split_ready = (
    X_train.shape == (72, 100, 6)
    and X_validation.shape == (24, 100, 6)
    and X_test.shape == (24, 100, 6)
    and train_counts == Counter({"left": 24, "right": 24, "up": 24})
    and validation_counts == Counter({"left": 8, "right": 8, "up": 8})
    and test_counts == Counter({"left": 8, "right": 8, "up": 8})
    and no_overlap
    and all_samples_assigned
)

print(f"RANDOM_SEED={RANDOM_SEED}")
print(f"TRAIN_SHAPE={X_train.shape}")
print(f"VALIDATION_SHAPE={X_validation.shape}")
print(f"TEST_SHAPE={X_test.shape}")
print(f"TRAIN_COUNTS={dict(train_counts)}")
print(f"VALIDATION_COUNTS={dict(validation_counts)}")
print(f"TEST_COUNTS={dict(test_counts)}")
print(f"NO_OVERLAP={no_overlap}")
print(f"ALL_SAMPLES_ASSIGNED={all_samples_assigned}")
print(f"SPLIT_READY={split_ready}")

RANDOM_SEED=42
TRAIN_SHAPE=(72, 100, 6)
VALIDATION_SHAPE=(24, 100, 6)
TEST_SHAPE=(24, 100, 6)
TRAIN_COUNTS={'left': 24, 'up': 24, 'right': 24}
VALIDATION_COUNTS={'left': 8, 'up': 8, 'right': 8}
TEST_COUNTS={'left': 8, 'up': 8, 'right': 8}
NO_OVERLAP=True
ALL_SAMPLES_ASSIGNED=True
SPLIT_READY=True


In [ ]:
import numpy as np

# Calculate normalization parameters using only the training split.
channel_mean = X_train.mean(axis=(0, 1))
channel_std = X_train.std(axis=(0, 1))

if np.any(channel_std == 0):
    zero_std_channels = [
        SENSOR_CHANNELS[index]
        for index in np.where(channel_std == 0)[0]
    ]
    raise ValueError(
        f"Zero standard deviation detected in channels: "
        f"{zero_std_channels}"
    )

X_train_normalized = (
    (X_train - channel_mean) / channel_std
).astype(np.float32)

X_validation_normalized = (
    (X_validation - channel_mean) / channel_std
).astype(np.float32)

X_test_normalized = (
    (X_test - channel_mean) / channel_std
).astype(np.float32)

training_normalized_mean = X_train_normalized.mean(axis=(0, 1))
training_normalized_std = X_train_normalized.std(axis=(0, 1))

normalization_ready = (
    np.isfinite(channel_mean).all()
    and np.isfinite(channel_std).all()
    and np.all(channel_std > 0)
    and np.isfinite(X_train_normalized).all()
    and np.isfinite(X_validation_normalized).all()
    and np.isfinite(X_test_normalized).all()
    and np.allclose(
        training_normalized_mean,
        np.zeros(len(SENSOR_CHANNELS)),
        atol=1e-5
    )
    and np.allclose(
        training_normalized_std,
        np.ones(len(SENSOR_CHANNELS)),
        atol=1e-5
    )
)

print("CHANNEL NORMALIZATION PARAMETERS")

for channel_index, channel_name in enumerate(SENSOR_CHANNELS):
    print(
        f"{channel_name}: "
        f"mean={channel_mean[channel_index]:.6f}, "
        f"std={channel_std[channel_index]:.6f}"
    )

print()
print(f"TRAIN_NORMALIZED_SHAPE={X_train_normalized.shape}")
print(
    "VALIDATION_NORMALIZED_SHAPE="
    f"{X_validation_normalized.shape}"
)
print(f"TEST_NORMALIZED_SHAPE={X_test_normalized.shape}")
print(
    "TRAIN_NORMALIZED_MEAN="
    f"{np.round(training_normalized_mean, 6).tolist()}"
)
print(
    "TRAIN_NORMALIZED_STD="
    f"{np.round(training_normalized_std, 6).tolist()}"
)
print(
    "ALL_NORMALIZED_VALUES_FINITE="
    f"{np.isfinite(X_train_normalized).all() and np.isfinite(X_validation_normalized).all() and np.isfinite(X_test_normalized).all()}"
)
print(f"NORMALIZATION_READY={normalization_ready}")

CHANNEL NORMALIZATION PARAMETERS
ax: mean=3142.514893, std=5776.777344
ay: mean=-571.772766, std=4798.743164
az: mean=20304.785156, std=3201.260498
gx: mean=104.275970, std=1553.918457
gy: mean=-578.386658, std=1474.625488
gz: mean=350.465271, std=3027.563477

TRAIN_NORMALIZED_SHAPE=(72, 100, 6)
VALIDATION_NORMALIZED_SHAPE=(24, 100, 6)
TEST_NORMALIZED_SHAPE=(24, 100, 6)
TRAIN_NORMALIZED_MEAN=[-0.0, -0.0, -3.000000106112566e-06, -0.0, -0.0, 0.0]
TRAIN_NORMALIZED_STD=[1.0000009536743164, 1.0, 1.0000020265579224, 1.0000009536743164, 1.0, 1.0]
ALL_NORMALIZED_VALUES_FINITE=True
NORMALIZATION_READY=True


In [ ]:
import numpy as np
import tensorflow as tf

tf.keras.backend.clear_session()
tf.keras.utils.set_random_seed(RANDOM_SEED)

model = tf.keras.Sequential(
    [
        tf.keras.layers.Input(
            shape=(
                SAMPLES_PER_CAPTURE,
                len(SENSOR_CHANNELS)
            ),
            name="mpu6050_input"
        ),
        tf.keras.layers.Conv1D(
            filters=16,
            kernel_size=5,
            activation="relu",
            name="conv1"
        ),
        tf.keras.layers.MaxPooling1D(
            pool_size=2,
            name="max_pool"
        ),
        tf.keras.layers.Conv1D(
            filters=32,
            kernel_size=3,
            activation="relu",
            name="conv2"
        ),
        tf.keras.layers.GlobalAveragePooling1D(
            name="global_average_pool"
        ),
        tf.keras.layers.Dense(
            units=16,
            activation="relu",
            name="dense_hidden"
        ),
        tf.keras.layers.Dense(
            units=len(GESTURE_LABELS),
            activation="softmax",
            name="gesture_output"
        )
    ],
    name="gesture_cnn_real_dataset"
)

model.compile(
    optimizer=tf.keras.optimizers.Adam(
        learning_rate=0.001
    ),
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)

early_stopping = tf.keras.callbacks.EarlyStopping(
    monitor="val_loss",
    patience=25,
    min_delta=0.0001,
    restore_best_weights=True,
    verbose=1
)

print(f"MODEL_INPUT_SHAPE={model.input_shape}")
print(f"MODEL_OUTPUT_SHAPE={model.output_shape}")
print(f"MODEL_PARAMETERS={model.count_params()}")

model.summary()

history = model.fit(
    X_train_normalized,
    y_train,
    validation_data=(
        X_validation_normalized,
        y_validation
    ),
    epochs=200,
    batch_size=12,
    shuffle=True,
    callbacks=[early_stopping],
    verbose=0
)

best_epoch = int(
    np.argmin(history.history["val_loss"]) + 1
)

training_loss, training_accuracy = model.evaluate(
    X_train_normalized,
    y_train,
    verbose=0
)

validation_loss, validation_accuracy = model.evaluate(
    X_validation_normalized,
    y_validation,
    verbose=0
)

model_ready = (
    model.input_shape == (None, 100, 6)
    and model.output_shape == (None, 3)
    and model.count_params() == 2643
    and np.isfinite(training_loss)
    and np.isfinite(validation_loss)
)

print()
print(f"TRAINED_EPOCHS={len(history.history['loss'])}")
print(f"BEST_EPOCH={best_epoch}")
print(f"TRAINING_LOSS={training_loss:.6f}")
print(f"TRAINING_ACCURACY={training_accuracy:.6f}")
print(f"VALIDATION_LOSS={validation_loss:.6f}")
print(f"VALIDATION_ACCURACY={validation_accuracy:.6f}")
print(f"MODEL_READY={model_ready}")

MODEL_INPUT_SHAPE=(None, 100, 6)
MODEL_OUTPUT_SHAPE=(None, 3)
MODEL_PARAMETERS=2643


Model: "gesture_cnn_real_dataset"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ conv1 (Conv1D)                  │ (None, 96, 16)         │           496 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pool (MaxPooling1D)         │ (None, 48, 16)         │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2 (Conv1D)                  │ (None, 46, 32)         │         1,568 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pool             │ (None, 32)             │             0 │
│ (GlobalAveragePooling1D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_hidden (Dense)            │ (None, 16)             │           528 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ gesture_output (Dense)          │ (None, 3)              │            51 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 2,643 (10.32 KB)

 Trainable params: 2,643 (10.32 KB)

 Non-trainable params: 0 (0.00 B)

Epoch 58: early stopping
Restoring model weights from the end of the best epoch: 33.

TRAINED_EPOCHS=58
BEST_EPOCH=33
TRAINING_LOSS=0.249407
TRAINING_ACCURACY=0.916667
VALIDATION_LOSS=0.302268
VALIDATION_ACCURACY=0.875000
MODEL_READY=True


In [ ]:
import numpy as np
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix
)

test_loss, test_accuracy = model.evaluate(
    X_test_normalized,
    y_test,
    verbose=0
)

test_probabilities = model.predict(
    X_test_normalized,
    verbose=0
)

test_predictions = np.argmax(
    test_probabilities,
    axis=1
)

calculated_test_accuracy = accuracy_score(
    y_test,
    test_predictions
)

test_confusion_matrix = confusion_matrix(
    y_test,
    test_predictions,
    labels=list(range(len(GESTURE_LABELS)))
)

print(f"TEST_LOSS={test_loss:.6f}")
print(f"TEST_ACCURACY={test_accuracy:.6f}")
print(
    "CALCULATED_TEST_ACCURACY="
    f"{calculated_test_accuracy:.6f}"
)

print("\nCONFUSION_MATRIX")
print(test_confusion_matrix)

print("\nCLASSIFICATION_REPORT")
print(
    classification_report(
        y_test,
        test_predictions,
        labels=list(range(len(GESTURE_LABELS))),
        target_names=GESTURE_LABELS,
        digits=4,
        zero_division=0
    )
)

correct_predictions = int(
    np.sum(test_predictions == y_test)
)

incorrect_predictions = int(
    np.sum(test_predictions != y_test)
)

test_ready = (
    len(test_predictions) == 24
    and test_confusion_matrix.shape == (3, 3)
    and np.isfinite(test_loss)
    and np.isfinite(test_probabilities).all()
    and np.isclose(
        test_accuracy,
        calculated_test_accuracy,
        atol=1e-6
    )
)

print(f"CORRECT_PREDICTIONS={correct_predictions}")
print(f"INCORRECT_PREDICTIONS={incorrect_predictions}")
print(f"TEST_READY={test_ready}")

TEST_LOSS=0.346130
TEST_ACCURACY=0.833333
CALCULATED_TEST_ACCURACY=0.833333

CONFUSION_MATRIX
[[8 0 0]
 [0 6 2]
 [0 2 6]]

CLASSIFICATION_REPORT
              precision    recall  f1-score   support

        left     1.0000    1.0000    1.0000         8
       right     0.7500    0.7500    0.7500         8
          up     0.7500    0.7500    0.7500         8

    accuracy                         0.8333        24
   macro avg     0.8333    0.8333    0.8333        24
weighted avg     0.8333    0.8333    0.8333        24

CORRECT_PREDICTIONS=20
INCORRECT_PREDICTIONS=4
TEST_READY=True


In [ ]:
import numpy as np

print("MISCLASSIFIED TEST CAPTURES")
print("=" * 100)

misclassified_indices = np.where(
    test_predictions != y_test
)[0]

for test_position in misclassified_indices:
    true_index = int(y_test[test_position])
    predicted_index = int(test_predictions[test_position])

    true_label = GESTURE_LABELS[true_index]
    predicted_label = GESTURE_LABELS[predicted_index]

    probabilities = test_probabilities[test_position]
    predicted_confidence = float(
        probabilities[predicted_index]
    )

    probability_text = ", ".join(
        f"{GESTURE_LABELS[index]}={probabilities[index]:.4f}"
        for index in range(len(GESTURE_LABELS))
    )

    print(f"FILE={test_file_names[test_position]}")
    print(f"TRUE_LABEL={true_label}")
    print(f"PREDICTED_LABEL={predicted_label}")
    print(f"PREDICTED_CONFIDENCE={predicted_confidence:.4f}")
    print(f"PROBABILITIES={probability_text}")
    print("-" * 100)

correct_confidences = np.max(
    test_probabilities[test_predictions == y_test],
    axis=1
)

incorrect_confidences = np.max(
    test_probabilities[test_predictions != y_test],
    axis=1
)

print()
print(f"MISCLASSIFIED_COUNT={len(misclassified_indices)}")
print(
    "MEAN_CORRECT_CONFIDENCE="
    f"{correct_confidences.mean():.4f}"
)
print(
    "MEAN_INCORRECT_CONFIDENCE="
    f"{incorrect_confidences.mean():.4f}"
)
print(
    "ERROR_ANALYSIS_READY="
    f"{len(misclassified_indices) == 4}"
)

MISCLASSIFIED TEST CAPTURES
FILE=20260730T221214_132875Z_up.json
TRUE_LABEL=up
PREDICTED_LABEL=right
PREDICTED_CONFIDENCE=0.6903
PROBABILITIES=left=0.0357, right=0.6903, up=0.2740
----------------------------------------------------------------------------------------------------
FILE=20260730T225522_153532Z_up.json
TRUE_LABEL=up
PREDICTED_LABEL=right
PREDICTED_CONFIDENCE=0.6642
PROBABILITIES=left=0.0617, right=0.6642, up=0.2741
----------------------------------------------------------------------------------------------------
FILE=20260730T232059_000907Z_right.json
TRUE_LABEL=right
PREDICTED_LABEL=up
PREDICTED_CONFIDENCE=0.5473
PROBABILITIES=left=0.0047, right=0.4480, up=0.5473
----------------------------------------------------------------------------------------------------
FILE=20260730T225126_231778Z_right.json
TRUE_LABEL=right
PREDICTED_LABEL=up
PREDICTED_CONFIDENCE=0.5020
PROBABILITIES=left=0.0101, right=0.4879, up=0.5020
-------------------------------------------------------

In [ ]:
import numpy as np

def relu(values):
    return np.maximum(values, 0.0)


def softmax(values):
    shifted_values = values - np.max(
        values,
        axis=1,
        keepdims=True
    )

    exponential_values = np.exp(shifted_values)

    return exponential_values / np.sum(
        exponential_values,
        axis=1,
        keepdims=True
    )


def conv1d_valid(inputs, kernel, bias):
    batch_size, input_length, input_channels = inputs.shape
    kernel_size, kernel_channels, output_channels = kernel.shape

    if input_channels != kernel_channels:
        raise ValueError(
            f"Conv1D channel mismatch: "
            f"input={input_channels}, kernel={kernel_channels}"
        )

    output_length = input_length - kernel_size + 1

    outputs = np.empty(
        (
            batch_size,
            output_length,
            output_channels
        ),
        dtype=np.float32
    )

    for output_position in range(output_length):
        input_window = inputs[
            :,
            output_position:output_position + kernel_size,
            :
        ]

        outputs[:, output_position, :] = (
            np.tensordot(
                input_window,
                kernel,
                axes=([1, 2], [0, 1])
            )
            + bias
        )

    return outputs


def max_pooling_1d(inputs, pool_size=2, stride=2):
    batch_size, input_length, channels = inputs.shape

    output_length = (
        (input_length - pool_size) // stride
    ) + 1

    outputs = np.empty(
        (
            batch_size,
            output_length,
            channels
        ),
        dtype=np.float32
    )

    for output_position in range(output_length):
        start_position = output_position * stride
        end_position = start_position + pool_size

        outputs[:, output_position, :] = np.max(
            inputs[:, start_position:end_position, :],
            axis=1
        )

    return outputs


conv1_kernel, conv1_bias = model.get_layer(
    "conv1"
).get_weights()

conv2_kernel, conv2_bias = model.get_layer(
    "conv2"
).get_weights()

dense_hidden_kernel, dense_hidden_bias = model.get_layer(
    "dense_hidden"
).get_weights()

gesture_output_kernel, gesture_output_bias = model.get_layer(
    "gesture_output"
).get_weights()


def numpy_model_predict(normalized_inputs):
    layer_output = conv1d_valid(
        normalized_inputs,
        conv1_kernel,
        conv1_bias
    )
    layer_output = relu(layer_output)

    layer_output = max_pooling_1d(
        layer_output,
        pool_size=2,
        stride=2
    )

    layer_output = conv1d_valid(
        layer_output,
        conv2_kernel,
        conv2_bias
    )
    layer_output = relu(layer_output)

    layer_output = np.mean(
        layer_output,
        axis=1
    )

    layer_output = (
        layer_output @ dense_hidden_kernel
        + dense_hidden_bias
    )
    layer_output = relu(layer_output)

    logits = (
        layer_output @ gesture_output_kernel
        + gesture_output_bias
    )

    return softmax(logits)


keras_test_probabilities = model.predict(
    X_test_normalized,
    verbose=0
)

numpy_test_probabilities = numpy_model_predict(
    X_test_normalized
)

keras_test_predictions = np.argmax(
    keras_test_probabilities,
    axis=1
)

numpy_test_predictions = np.argmax(
    numpy_test_probabilities,
    axis=1
)

maximum_probability_difference = float(
    np.max(
        np.abs(
            keras_test_probabilities
            - numpy_test_probabilities
        )
    )
)

prediction_matches = np.array_equal(
    keras_test_predictions,
    numpy_test_predictions
)

probability_matches = np.allclose(
    keras_test_probabilities,
    numpy_test_probabilities,
    atol=1e-5,
    rtol=1e-5
)

numpy_accuracy = float(
    np.mean(numpy_test_predictions == y_test)
)

numpy_inference_ready = (
    prediction_matches
    and probability_matches
    and np.isclose(
        numpy_accuracy,
        test_accuracy,
        atol=1e-6
    )
)

print(
    "KERAS_PROBABILITY_SHAPE="
    f"{keras_test_probabilities.shape}"
)
print(
    "NUMPY_PROBABILITY_SHAPE="
    f"{numpy_test_probabilities.shape}"
)
print(
    "MAXIMUM_PROBABILITY_DIFFERENCE="
    f"{maximum_probability_difference:.10f}"
)
print(f"PREDICTION_MATCHES={prediction_matches}")
print(f"PROBABILITY_MATCHES={probability_matches}")
print(f"KERAS_TEST_ACCURACY={test_accuracy:.6f}")
print(f"NUMPY_TEST_ACCURACY={numpy_accuracy:.6f}")
print(f"NUMPY_INFERENCE_READY={numpy_inference_ready}")

KERAS_PROBABILITY_SHAPE=(24, 3)
NUMPY_PROBABILITY_SHAPE=(24, 3)
MAXIMUM_PROBABILITY_DIFFERENCE=0.0000001788
PREDICTION_MATCHES=True
PROBABILITY_MATCHES=True
KERAS_TEST_ACCURACY=0.833333
NUMPY_TEST_ACCURACY=0.833333
NUMPY_INFERENCE_READY=True


In [ ]:
from pathlib import Path
import hashlib
import json
import shutil
import numpy as np
import tensorflow as tf

PACKAGE_NAME = "gesture_model_real_v1"
PACKAGE_PATH = Path("/content") / PACKAGE_NAME
ZIP_PATH = Path("/content") / f"{PACKAGE_NAME}.zip"

if PACKAGE_PATH.exists():
    shutil.rmtree(PACKAGE_PATH)

if ZIP_PATH.exists():
    ZIP_PATH.unlink()

PACKAGE_PATH.mkdir(parents=True, exist_ok=True)

# ---------------------------------------------------------
# 1. Save the trained Keras model
# ---------------------------------------------------------

keras_model_path = PACKAGE_PATH / "gesture_cnn_real.keras"
model.save(keras_model_path)

# ---------------------------------------------------------
# 2. Convert and save the TensorFlow Lite model
# ---------------------------------------------------------

tflite_converter = tf.lite.TFLiteConverter.from_keras_model(model)
tflite_model = tflite_converter.convert()

tflite_model_path = PACKAGE_PATH / "gesture_cnn_real.tflite"
tflite_model_path.write_bytes(tflite_model)

# ---------------------------------------------------------
# 3. Save weights for manual NumPy inference
# ---------------------------------------------------------

numpy_weights_path = (
    PACKAGE_PATH / "gesture_cnn_numpy_weights.npz"
)

np.savez_compressed(
    numpy_weights_path,
    conv1_kernel=conv1_kernel.astype(np.float32),
    conv1_bias=conv1_bias.astype(np.float32),
    conv2_kernel=conv2_kernel.astype(np.float32),
    conv2_bias=conv2_bias.astype(np.float32),
    dense_hidden_kernel=dense_hidden_kernel.astype(np.float32),
    dense_hidden_bias=dense_hidden_bias.astype(np.float32),
    gesture_output_kernel=gesture_output_kernel.astype(np.float32),
    gesture_output_bias=gesture_output_bias.astype(np.float32)
)

# ---------------------------------------------------------
# 4. Save normalization parameters
# ---------------------------------------------------------

normalization_parameters = {
    "channels": SENSOR_CHANNELS,
    "mean": [
        float(value)
        for value in channel_mean.astype(np.float32)
    ],
    "standard_deviation": [
        float(value)
        for value in channel_std.astype(np.float32)
    ],
    "calculated_from": "training_split_only",
    "training_capture_count": int(len(X_train))
}

normalization_path = (
    PACKAGE_PATH / "normalization_parameters.json"
)

normalization_path.write_text(
    json.dumps(
        normalization_parameters,
        indent=2
    ),
    encoding="utf-8"
)

# ---------------------------------------------------------
# 5. Save labels
# ---------------------------------------------------------

labels_data = {
    "labels": GESTURE_LABELS,
    "label_to_index": label_to_index,
    "index_to_label": {
        str(index): label
        for index, label in enumerate(GESTURE_LABELS)
    }
}

labels_path = PACKAGE_PATH / "gesture_labels.json"

labels_path.write_text(
    json.dumps(labels_data, indent=2),
    encoding="utf-8"
)

# ---------------------------------------------------------
# 6. Save model configuration
# ---------------------------------------------------------

model_configuration = {
    "project_name": (
        "Wireless Embedded Gesture Recognition System"
    ),
    "model_version": "real_v1",
    "model_type": "CNN_1D",
    "input_shape": [100, 6],
    "output_classes": GESTURE_LABELS,
    "sensor_channels": SENSOR_CHANNELS,
    "sample_rate_hz": int(
        dataset_config["sample_rate_hz"]
    ),
    "samples_per_capture": int(
        dataset_config["samples_per_capture"]
    ),
    "architecture": [
        {
            "type": "Conv1D",
            "filters": 16,
            "kernel_size": 5,
            "activation": "relu",
            "padding": "valid"
        },
        {
            "type": "MaxPooling1D",
            "pool_size": 2,
            "stride": 2
        },
        {
            "type": "Conv1D",
            "filters": 32,
            "kernel_size": 3,
            "activation": "relu",
            "padding": "valid"
        },
        {
            "type": "GlobalAveragePooling1D"
        },
        {
            "type": "Dense",
            "units": 16,
            "activation": "relu"
        },
        {
            "type": "Dense",
            "units": 3,
            "activation": "softmax"
        }
    ],
    "parameter_count": int(model.count_params()),
    "inference_engine_for_raspberry": "numpy_cnn_1d"
}

model_config_path = PACKAGE_PATH / "model_config.json"

model_config_path.write_text(
    json.dumps(model_configuration, indent=2),
    encoding="utf-8"
)

# ---------------------------------------------------------
# 7. Save evaluation metrics
# ---------------------------------------------------------

evaluation_metrics = {
    "random_seed": int(RANDOM_SEED),
    "dataset_capture_count": int(len(X)),
    "training_capture_count": int(len(X_train)),
    "validation_capture_count": int(len(X_validation)),
    "test_capture_count": int(len(X_test)),
    "best_epoch": int(best_epoch),
    "trained_epochs": int(len(history.history["loss"])),
    "training_accuracy": float(training_accuracy),
    "validation_accuracy": float(validation_accuracy),
    "test_accuracy": float(test_accuracy),
    "test_loss": float(test_loss),
    "correct_test_predictions": int(correct_predictions),
    "incorrect_test_predictions": int(incorrect_predictions),
    "confusion_matrix": test_confusion_matrix.tolist(),
    "keras_numpy_maximum_probability_difference": (
        maximum_probability_difference
    ),
    "keras_numpy_predictions_match": bool(
        prediction_matches
    ),
    "keras_numpy_probabilities_match": bool(
        probability_matches
    )
}

evaluation_path = PACKAGE_PATH / "evaluation_metrics.json"

evaluation_path.write_text(
    json.dumps(evaluation_metrics, indent=2),
    encoding="utf-8"
)

# ---------------------------------------------------------
# 8. Save dataset split manifest
# ---------------------------------------------------------

split_manifest = {
    "random_seed": int(RANDOM_SEED),
    "training_files": train_file_names.tolist(),
    "validation_files": validation_file_names.tolist(),
    "test_files": test_file_names.tolist()
}

split_manifest_path = PACKAGE_PATH / "split_manifest.json"

split_manifest_path.write_text(
    json.dumps(split_manifest, indent=2),
    encoding="utf-8"
)

# ---------------------------------------------------------
# 9. Save training history
# ---------------------------------------------------------

training_history = {
    metric_name: [
        float(value)
        for value in metric_values
    ]
    for metric_name, metric_values
    in history.history.items()
}

training_history_path = (
    PACKAGE_PATH / "training_history.json"
)

training_history_path.write_text(
    json.dumps(training_history, indent=2),
    encoding="utf-8"
)

# ---------------------------------------------------------
# 10. Create artifact manifest with SHA-256 hashes
# ---------------------------------------------------------

artifact_files = sorted(
    file_path
    for file_path in PACKAGE_PATH.iterdir()
    if file_path.is_file()
)

artifact_manifest = {
    "package_name": PACKAGE_NAME,
    "model_version": "real_v1",
    "artifacts": {}
}

for artifact_path in artifact_files:
    artifact_hash = hashlib.sha256(
        artifact_path.read_bytes()
    ).hexdigest().upper()

    artifact_manifest["artifacts"][artifact_path.name] = {
        "size_bytes": int(artifact_path.stat().st_size),
        "sha256": artifact_hash
    }

artifact_manifest_path = (
    PACKAGE_PATH / "artifact_manifest.json"
)

artifact_manifest_path.write_text(
    json.dumps(artifact_manifest, indent=2),
    encoding="utf-8"
)

# ---------------------------------------------------------
# 11. Create ZIP package
# ---------------------------------------------------------

shutil.make_archive(
    base_name=str(ZIP_PATH.with_suffix("")),
    format="zip",
    root_dir="/content",
    base_dir=PACKAGE_NAME
)

required_files = {
    "gesture_cnn_real.keras",
    "gesture_cnn_real.tflite",
    "gesture_cnn_numpy_weights.npz",
    "normalization_parameters.json",
    "gesture_labels.json",
    "model_config.json",
    "evaluation_metrics.json",
    "split_manifest.json",
    "training_history.json",
    "artifact_manifest.json"
}

created_files = {
    file_path.name
    for file_path in PACKAGE_PATH.iterdir()
    if file_path.is_file()
}

package_ready = (
    required_files.issubset(created_files)
    and ZIP_PATH.exists()
    and ZIP_PATH.stat().st_size > 0
)

zip_sha256 = hashlib.sha256(
    ZIP_PATH.read_bytes()
).hexdigest().upper()

print(f"PACKAGE_PATH={PACKAGE_PATH}")
print(f"PACKAGE_FILE_COUNT={len(created_files)}")
print(f"REQUIRED_FILES_PRESENT={required_files.issubset(created_files)}")
print(f"ZIP_PATH={ZIP_PATH}")
print(f"ZIP_SIZE_KB={ZIP_PATH.stat().st_size / 1024:.1f}")
print(f"ZIP_SHA256={zip_sha256}")
print(f"PACKAGE_READY={package_ready}")

Saved artifact at '/tmp/tmpl70equ3w'. The following endpoints are available:

* Endpoint 'serve'
  args_0 (POSITIONAL_ONLY): TensorSpec(shape=(None, 100, 6), dtype=tf.float32, name='mpu6050_input')
Output Type:
  TensorSpec(shape=(None, 3), dtype=tf.float32, name=None)
Captures:
  138081217623248: TensorSpec(shape=(), dtype=tf.resource, name=None)
  138081217624400: TensorSpec(shape=(), dtype=tf.resource, name=None)
  138081217623056: TensorSpec(shape=(), dtype=tf.resource, name=None)
  138081217621712: TensorSpec(shape=(), dtype=tf.resource, name=None)
  138081217623824: TensorSpec(shape=(), dtype=tf.resource, name=None)
  138081217623440: TensorSpec(shape=(), dtype=tf.resource, name=None)
  138081217624784: TensorSpec(shape=(), dtype=tf.resource, name=None)
  138081217625936: TensorSpec(shape=(), dtype=tf.resource, name=None)
PACKAGE_PATH=/content/gesture_model_real_v1
PACKAGE_FILE_COUNT=10
REQUIRED_FILES_PRESENT=True
ZIP_PATH=/content/gesture_model_real_v1.zip
ZIP_SIZE_KB=60.2
ZIP_S

In [ ]:
from google.colab import files

files.download("/content/gesture_model_real_v1.zip")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>